# Análise Financeira com Python

Este notebook:

1. **Lê e valida** o arquivo `transacoes.csv` (módulo `csv` nativo, sem pandas).
2. **Limpa** os dados, descartando silenciosamente linhas inválidas.
3. **Agrupa por mês** e calcula métricas financeiras.
4. **Sinaliza** transações suspeitas (valor acima de `R$ 10.000,00`).
5. **Exibe** um relatório formatado.
6. **Exporta** o resultado em `relatorio.json`.


## Criação do arquivo de dados (`transacoes.csv`)

A célula abaixo gera o `transacoes.csv`.

O dataset contém:
- **18 registros válidos** distribuídos em **4 meses** (2026-01 a 2026-04);
- **7 registros inválidos**, um para cada regra de validação;
- **3 transações acima de R\$ 10.000,00** (suspeitas).

In [1]:
# Conteúdo do CSV de teste. Cada linha inválida está comentada com o motivo.
csv_conteudo = """id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salario janeiro,salario
2,2026-01-12,CLI002,debito,180.50,Supermercado,compra
3,2026-01-18,CLI001,debito,1200.00,Aluguel,moradia
4,2026-01-25,CLI003,credito,12500.00,Bonus anual,salario
5,2026-01-28,CLI002,debito,89.90,Streaming,assinatura
6,2026-02-03,CLI001,credito,3500.00,Salario fevereiro,salario
7,2026-02-10,CLI003,debito,450.00,Conta de luz,conta
8,2026-02-14,CLI002,credito,15000.00,Transferencia recebida,transferencia
9,2026-02-18,CLI001,debito,320.00,Farmacia,compra
10,2026-02-22,CLI003,debito,99.90,Streaming,assinatura
11,2026-03-01,CLI001,credito,3500.00,Salario marco,salario
12,2026-03-08,CLI002,debito,760.00,Mercado mensal,compra
13,2026-03-15,CLI003,credito,2200.00,Freelance,transferencia
14,2026-03-20,CLI001,debito,1200.00,Aluguel,moradia
15,2026-03-27,CLI002,debito,230.45,Restaurante,compra
16,2026-04-02,CLI003,credito,3800.00,Salario abril,salario
17,2026-04-09,CLI001,debito,540.00,Internet e telefone,conta
18,2026-04-15,CLI002,debito,11200.00,Compra de carro usado,compra
,2026-04-20,CLI001,debito,150.00,Linha sem id,compra
abc,2026-04-21,CLI002,credito,200.00,Id nao numerico,salario
21,2026-04-22,,debito,200.00,Cliente vazio,transferencia
22,20-04-2026,CLI003,debito,75.00,Data mal formatada,compra
23,2026-04-23,CLI001,pix,500.00,Tipo invalido,transferencia
24,2026-04-24,CLI002,debito,abc,Valor nao numerico,compra
25,2026-04-25,CLI003,credito,-50.00,Valor negativo,salario
"""

with open("transacoes.csv", "w", encoding="utf-8") as arquivo:
    arquivo.write(csv_conteudo)

print("Arquivo 'transacoes.csv' criado.")
print(f"Total de linhas de dados: {len(csv_conteudo.strip().splitlines()) - 1}")

Arquivo 'transacoes.csv' criado.
Total de linhas de dados: 25


## Configurações e constantes

Imports e constantes globais usadas pelo restante do notebook.

In [2]:
import csv
import json
from datetime import datetime, date

# --- Constantes de configuracao ---
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
LIMITE_SUSPEITO = 10000.00   # transacoes acima deste valor sao sinalizadas
TIPOS_VALIDOS = ("credito", "debito")
FORMATO_DATA = "%Y-%m-%d"    # AAAA-MM-DD

print("Configuracoes carregadas.")
print(f"  Arquivo de entrada : {ARQUIVO_CSV}")
print(f"  Arquivo de saida   : {ARQUIVO_JSON}")
print(f"  Limite suspeito    : R$ {LIMITE_SUSPEITO:,.2f}")

Configuracoes carregadas.
  Arquivo de entrada : transacoes.csv
  Arquivo de saida   : relatorio.json
  Limite suspeito    : R$ 10,000.00


## Leitura do CSV (`ler_transacoes`)

Usa o módulo **`csv` nativo** com `csv.DictReader` para acessar colunas pelo nome.
O `try/except` (situação 1 de 3) captura `FileNotFoundError` caso o arquivo não exista,
sem encerrar o programa.

In [3]:
def ler_transacoes(caminho):
    """Le o CSV e retorna a lista de transacoes brutas (dicts).

    Em caso de arquivo inexistente, avisa e retorna lista vazia.
    """
    try:
        with open(caminho, newline="", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        print(f"[ERRO] Arquivo '{caminho}' nao encontrado.")
        return []


# --- Teste rapido ---
_brutas = ler_transacoes(ARQUIVO_CSV)
print(f"Linhas lidas: {len(_brutas)}")
print("Primeira linha:", _brutas[0])
print("Arquivo inexistente ->", ler_transacoes("nao_existe.csv"))

Linhas lidas: 25
Primeira linha: {'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salario janeiro', 'categoria': 'salario'}
[ERRO] Arquivo 'nao_existe.csv' nao encontrado.
Arquivo inexistente -> []


## Validação e limpeza dos dados

Funções pequenas e reutilizáveis, uma responsabilidade cada:

- `validar_data` → `try/except` com `strptime` (situação 2 de 3);
- `validar_valor` → `try/except` na conversão para `float` (situação 3 de 3);
- `validar_transacao` → aplica todas as regras e devolve o registro limpo (ou `None`).

Regras para **descartar** a linha: `id` vazio/não numérico · `cliente_id` vazio ·
`data` em formato inválido · `tipo` diferente de *credito/debito* · `valor` não numérico ou ≤ 0.

In [4]:
def validar_data(texto):
    """Converte 'AAAA-MM-DD' em datetime. Retorna None se invalido."""
    try:
        return datetime.strptime(texto, FORMATO_DATA)
    except (ValueError, TypeError):
        return None


def validar_valor(texto):
    """Converte texto em float. Retorna None se nao for numerico."""
    try:
        return float(texto)
    except (ValueError, TypeError):
        return None


def validar_transacao(linha):
    """Valida uma unica linha e retorna o registro limpo, ou None se invalida."""
    id_txt = (linha.get("id") or "").strip()
    if not id_txt.isdigit():                       # id vazio ou nao numerico
        return None

    cliente = (linha.get("cliente_id") or "").strip()
    if not cliente:                                # cliente_id vazio
        return None

    data_obj = validar_data((linha.get("data") or "").strip())
    if data_obj is None:                           # data mal formatada
        return None

    tipo = (linha.get("tipo") or "").strip().lower()
    if tipo not in TIPOS_VALIDOS:                  # tipo invalido
        return None

    valor = validar_valor((linha.get("valor") or "").strip())
    if valor is None or valor <= 0:                # valor invalido ou <= 0
        return None

    return {
        "id": int(id_txt),
        "data": data_obj,
        "mes": data_obj.strftime("%Y-%m"),
        "cliente_id": cliente,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip(),
    }

In [5]:
# --- Teste rapido da validacao ---
print("Data valida  :", validar_data("2026-01-05"))
print("Data invalida:", validar_data("20-04-2026"))
print("Valor valido :", validar_valor("3500.00"))
print("Valor invalido:", validar_valor("abc"))
print()

linha_ok = {"id": "1", "data": "2026-01-05", "cliente_id": "CLI001",
            "tipo": "credito", "valor": "3500.00", "descricao": "Salario", "categoria": "salario"}
linha_ruim = {"id": "", "data": "2026-01-05", "cliente_id": "CLI001",
              "tipo": "credito", "valor": "3500.00", "descricao": "", "categoria": ""}
print("Linha valida  ->", validar_transacao(linha_ok))
print("Linha invalida ->", validar_transacao(linha_ruim))

Data valida  : 2026-01-05 00:00:00
Data invalida: None
Valor valido : 3500.0
Valor invalido: None

Linha valida  -> {'id': 1, 'data': datetime.datetime(2026, 1, 5, 0, 0), 'mes': '2026-01', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': 3500.0, 'descricao': 'Salario', 'categoria': 'salario'}
Linha invalida -> None


## Manipulação de datas

`calcular_periodo` usa os objetos `datetime` para descobrir a transação mais antiga,
a mais recente e quantos **dias** se passaram entre elas.

In [6]:
def calcular_periodo(transacoes):
    """Retorna data mais antiga, mais recente e o intervalo em dias."""
    datas = [t["data"] for t in transacoes]
    mais_antiga = min(datas)
    mais_recente = max(datas)
    return {
        "data_mais_antiga": mais_antiga.strftime(FORMATO_DATA),
        "data_mais_recente": mais_recente.strftime(FORMATO_DATA),
        "dias_no_periodo": (mais_recente - mais_antiga).days,
    }


# --- Teste rapido ---
_validas_teste = [r for r in (validar_transacao(l) for l in _brutas) if r]
print(calcular_periodo(_validas_teste))

{'data_mais_antiga': '2026-01-05', 'data_mais_recente': '2026-04-15', 'dias_no_periodo': 100}


## Agrupamento mensal e métricas (`gerar_relatorio`)

Para cada mês (`AAAA-MM`) calcula: quantidade, total de crédito, total de débito,
saldo (crédito − débito), valor médio, maior e menor valor. Também monta a lista de
**transações suspeitas** (valor acima de `LIMITE_SUSPEITO`) e o cabeçalho do relatório.

In [7]:
def _resumir_mes(valores, total_credito, total_debito):
    """Calcula as metricas de um unico mes a partir dos acumuladores."""
    return {
        "quantidade": len(valores),
        "total_credito": round(total_credito, 2),
        "total_debito": round(total_debito, 2),
        "saldo": round(total_credito - total_debito, 2),
        "media": round(sum(valores) / len(valores), 2),
        "maior_valor": round(max(valores), 2),
        "menor_valor": round(min(valores), 2),
    }


def gerar_relatorio(transacoes, total_invalidas):
    """Agrupa as transacoes por mes, calcula metricas e monta o relatorio final."""
    acumulado = {}      # mes -> {valores, credito, debito}
    suspeitas = []

    for t in transacoes:
        mes = t["mes"]
        if mes not in acumulado:
            acumulado[mes] = {"valores": [], "credito": 0.0, "debito": 0.0}
        acumulado[mes]["valores"].append(t["valor"])
        if t["tipo"] == "credito":
            acumulado[mes]["credito"] += t["valor"]
        else:
            acumulado[mes]["debito"] += t["valor"]

        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime(FORMATO_DATA),
                "valor": round(t["valor"], 2),
            })

    resumo_mensal = {
        mes: _resumir_mes(d["valores"], d["credito"], d["debito"])
        for mes, d in sorted(acumulado.items())
    }

    return {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": total_invalidas,
        "periodo": calcular_periodo(transacoes),
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": suspeitas,
    }


# --- Teste rapido ---
_rel = gerar_relatorio(_validas_teste, total_invalidas=7)
print("Meses:", list(_rel["resumo_mensal"].keys()))
print("Jan/2026:", _rel["resumo_mensal"]["2026-01"])
print("Suspeitas:", len(_rel["transacoes_suspeitas"]))

Meses: ['2026-01', '2026-02', '2026-03', '2026-04']
Jan/2026: {'quantidade': 5, 'total_credito': 16000.0, 'total_debito': 1470.4, 'saldo': 14529.6, 'media': 3494.08, 'maior_valor': 12500.0, 'menor_valor': 89.9}
Suspeitas: 3


## Formatação e exibição no terminal (`exibir_relatorio`)

`formatar_real` aplica o padrão monetário brasileiro (`R$ 3.500,00`).
`exibir_relatorio` imprime as seções com separadores visuais.

In [8]:
def formatar_real(valor):
    """Formata um numero no padrao monetario brasileiro: R$ 1.234,56."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio):
    """Imprime o relatorio formatado no terminal."""
    periodo = relatorio["periodo"]
    print("=" * 40)
    print("        RELATORIO FINANCEIRO CLEARBANK")
    print("=" * 40)
    print(f"Periodo analisado : {periodo['data_mais_antiga']} -> {periodo['data_mais_recente']}")
    print(f"Dias no periodo   : {periodo['dias_no_periodo']}")
    print(f"Transacoes validas  : {relatorio['total_transacoes_validas']}")
    print(f"Transacoes invalidas: {relatorio['total_transacoes_invalidas']}")

    print("\n===== RELATORIO MENSAL =====")
    for mes, m in relatorio["resumo_mensal"].items():
        print(f"\nMes: {mes}")
        print(f"  Transacoes:    {m['quantidade']}")
        print(f"  Total credito: {formatar_real(m['total_credito'])}")
        print(f"  Total debito:  {formatar_real(m['total_debito'])}")
        print(f"  Saldo:         {formatar_real(m['saldo'])}")
        print(f"  Media:         {formatar_real(m['media'])}")
        print(f"  Maior valor:   {formatar_real(m['maior_valor'])}")
        print(f"  Menor valor:   {formatar_real(m['menor_valor'])}")

    print("\n===== TRANSACOES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transacao suspeita encontrada.")
    else:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | "
                  f"Data: {s['data']} | Valor: {formatar_real(s['valor'])}")


# --- Teste rapido ---
print(formatar_real(3500.0), "|", formatar_real(180.5), "|", formatar_real(15000.0))

R$ 3.500,00 | R$ 180,50 | R$ 15.000,00


## Exportação do relatório em JSON (`salvar_json`)

Usa `json.dump(..., ensure_ascii=False, indent=2)` para gerar um JSON legível e com
acentos corretos. O `try/except` aqui (4ª situação, além do mínimo de 3) protege contra
erros de escrita em disco.

In [9]:
def salvar_json(relatorio, caminho):
    """Salva o relatorio no arquivo JSON indicado."""
    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
        print(f"[OK] Relatorio salvo em '{caminho}'.")
    except OSError as erro:
        print(f"[ERRO] Falha ao salvar JSON: {erro}")


# --- Teste rapido ---
salvar_json(_rel, ARQUIVO_JSON)
with open(ARQUIVO_JSON, encoding="utf-8") as f:
    print(f.read()[:400], "...")

[OK] Relatorio salvo em 'relatorio.json'.
{
  "gerado_em": "2026-07-08",
  "total_transacoes_validas": 18,
  "total_transacoes_invalidas": 7,
  "periodo": {
    "data_mais_antiga": "2026-01-05",
    "data_mais_recente": "2026-04-15",
    "dias_no_periodo": 100
  },
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 5,
      "total_credito": 16000.0,
      "total_debito": 1470.4,
      "saldo": 14529.6,
      "media": 3494.08,
      ...


## Execução Principal

Junta todas as etapas: leitura → validação/limpeza (com resumo) → métricas →
exibição → exportação JSON. Erros em linhas individuais são capturados nas funções
de validação, então a linha é descartada e o processamento continua.

In [10]:
def main():
    # 1) Leitura
    linhas = ler_transacoes(ARQUIVO_CSV)
    if not linhas:
        print("Nada a processar.")
        return

    # 2) Validacao e limpeza
    validas = []
    invalidas = 0
    for linha in linhas:
        registro = validar_transacao(linha)
        if registro is None:
            invalidas += 1          # descarte silencioso
        else:
            validas.append(registro)

    print("===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {len(linhas)}")
    print(f"Linhas validas: {len(validas)}")
    print(f"Linhas invalidas: {invalidas}\n")

    if not validas:
        print("Nenhuma transacao valida para processar.")
        return

    # 3) Metricas  4) Exibicao  5) Exportacao
    relatorio = gerar_relatorio(validas, invalidas)
    exibir_relatorio(relatorio)
    print()
    salvar_json(relatorio, ARQUIVO_JSON)


main()

===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 25
Linhas validas: 18
Linhas invalidas: 7

        RELATORIO FINANCEIRO CLEARBANK
Periodo analisado : 2026-01-05 -> 2026-04-15
Dias no periodo   : 100
Transacoes validas  : 18
Transacoes invalidas: 7

===== RELATORIO MENSAL =====

Mes: 2026-01
  Transacoes:    5
  Total credito: R$ 16.000,00
  Total debito:  R$ 1.470,40
  Saldo:         R$ 14.529,60
  Media:         R$ 3.494,08
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 89,90

Mes: 2026-02
  Transacoes:    5
  Total credito: R$ 18.500,00
  Total debito:  R$ 869,90
  Saldo:         R$ 17.630,10
  Media:         R$ 3.873,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mes: 2026-03
  Transacoes:    5
  Total credito: R$ 5.700,00
  Total debito:  R$ 2.190,45
  Saldo:         R$ 3.509,55
  Media:         R$ 1.578,09
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 230,45

Mes: 2026-04
  Transacoes:    3
  Total credito: R$ 3.800,00
  Total debito:  R$ 11.740,00
  Saldo:

## Análise com `pandas`

Versão alternativa da leitura e do agrupamento usando `pandas`, para **conferir** se os
valores batem com a solução nativa. A versão completa também está no arquivo
`analise_pandas.py` do repositório.

In [11]:
import pandas as pd

# Leitura e limpeza equivalente (mesmas regras da solucao nativa)
df = pd.read_csv(ARQUIVO_CSV, dtype=str)
df["valor_num"] = pd.to_numeric(df["valor"], errors="coerce")
df["data_dt"] = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")

valido = (
    df["id"].str.fullmatch(r"\d+", na=False)
    & df["cliente_id"].str.strip().ne("") & df["cliente_id"].notna()
    & df["data_dt"].notna()
    & df["tipo"].str.lower().isin(["credito", "debito"])
    & df["valor_num"].notna() & (df["valor_num"] > 0)
)
limpo = df[valido].copy()
limpo["mes"] = limpo["data_dt"].dt.strftime("%Y-%m")
limpo["tipo"] = limpo["tipo"].str.lower()

def metricas(grupo):
    cred = grupo.loc[grupo["tipo"] == "credito", "valor_num"].sum()
    deb = grupo.loc[grupo["tipo"] == "debito", "valor_num"].sum()
    return pd.Series({
        "quantidade": len(grupo),
        "total_credito": round(cred, 2),
        "total_debito": round(deb, 2),
        "saldo": round(cred - deb, 2),
        "media": round(grupo["valor_num"].mean(), 2),
        "maior_valor": round(grupo["valor_num"].max(), 2),
        "menor_valor": round(grupo["valor_num"].min(), 2),
    })

resumo_pd = limpo.groupby("mes").apply(metricas, include_groups=False)
print(resumo_pd)
print(f"\nValidas (pandas): {len(limpo)} | Invalidas: {len(df) - len(limpo)}")

# Conferencia automatica contra a solucao nativa
nativo = gerar_relatorio([r for r in (validar_transacao(l) for l in ler_transacoes(ARQUIVO_CSV)) if r], 0)
iguais = all(
    abs(resumo_pd.loc[mes, "saldo"] - nativo["resumo_mensal"][mes]["saldo"]) < 1e-6
    for mes in nativo["resumo_mensal"]
)
print("Saldos batem com a solucao nativa?", iguais)

         quantidade  total_credito  total_debito     saldo    media  \
mes                                                                   
2026-01         5.0        16000.0       1470.40  14529.60  3494.08   
2026-02         5.0        18500.0        869.90  17630.10  3873.98   
2026-03         5.0         5700.0       2190.45   3509.55  1578.09   
2026-04         3.0         3800.0      11740.00  -7940.00  5180.00   

         maior_valor  menor_valor  
mes                                
2026-01      12500.0        89.90  
2026-02      15000.0        99.90  
2026-03       3500.0       230.45  
2026-04      11200.0       540.00  

Validas (pandas): 18 | Invalidas: 7
Saldos batem com a solucao nativa? True


## Visualização com `matplotlib`

Gráfico de barras com o **saldo mensal** (crédito − débito) salvo como `grafico.png`,
com título, rótulos nos eixos e linha de referência no zero.

In [12]:
import matplotlib
matplotlib.use("Agg")  # backend sem display (Colab/Jupyter geram a imagem igual)
import matplotlib.pyplot as plt

relatorio = gerar_relatorio(
    [r for r in (validar_transacao(l) for l in ler_transacoes(ARQUIVO_CSV)) if r], 0
)
meses = list(relatorio["resumo_mensal"].keys())
saldos = [relatorio["resumo_mensal"][m]["saldo"] for m in meses]

fig, ax = plt.subplots(figsize=(8, 4.5))
cores = ["#2e8b57" if s >= 0 else "#c0392b" for s in saldos]
barras = ax.bar(meses, saldos, color=cores, label="Saldo (credito - debito)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Saldo mensal — ClearBank")
ax.set_xlabel("Mes")
ax.set_ylabel("Saldo (R$)")
ax.legend()
for barra, s in zip(barras, saldos):
    ax.text(barra.get_x() + barra.get_width() / 2, s,
            f"R$ {s:,.0f}", ha="center",
            va="bottom" if s >= 0 else "top", fontsize=9)
plt.tight_layout()
plt.savefig("grafico.png", dpi=120)
plt.show()
print("Grafico salvo em 'grafico.png'.")

Grafico salvo em 'grafico.png'.


C:\Users\f\AppData\Local\Temp\ipykernel_5772\943908526.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
